In [2]:
# This script:
# 1. Loads the Step 1 filtered dataset
# 2. Drops variables with >88% structural missingness (ALCOCCAS, ALCFREQ, INKNOWN)
# 3. Handles smoking variables — structural -4 (never smoked) vs true missing
# 4. Recodes all sentinel codes to NaN per variable
# 5. Handles INVISITS / INCALLS structural missingness (8 = lives with co-participant)
# 6. Applies baseline-only visit strategy (one row per subject, first visit)
# 7. Produces a clean dataset ready for feature engineering (Step 3)

import pandas as pd
import numpy as np
import os
 
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 200)

In [3]:
INPUT_PATH  = r"C:\Users\USER\PycharmProjects\Non-med-dementia-risk\outputs\step1_filtered_dataset.csv"
OUTPUT_DIR  = r"C:\Users\USER\PycharmProjects\Non-med-dementia-risk\outputs"
ID_COL      = "NACCID"
LABEL_COL   = "dementia_binary"
 
df = pd.read_csv(INPUT_PATH, low_memory=False)
print(f"Loaded: {df.shape}")

Loaded: (195196, 29)


In [4]:
# 1. Drop variables with >88% structural missingness
# ALCOCCAS and ALCFREQ: ~90% of rows are -4 (form version did not collect this).
# INKNOWN: ~88% of rows are -4. Retaining these would mean imputing nearly the
# entire column — the signal-to-noise ratio is too low to be useful.
 

DROP_COLS = ["ALCOCCAS", "ALCFREQ", "INKNOWN"]
df = df.drop(columns=[c for c in DROP_COLS if c in df.columns])
print(f"Dropped {DROP_COLS}. Shape now: {df.shape}")

Dropped ['ALCOCCAS', 'ALCFREQ', 'INKNOWN']. Shape now: (195196, 26)


In [5]:
# 2. Smoking variables — structural missingness vs true missing
#
# Exactly 71,664 rows have -4 across all smoking variables (TOBAC30, TOBAC100,
# SMOKYRS, PACKSPER, QUITSMOK). These are subjects for whom the follow-up smoking
# questions were skipped because they indicated they had never smoked.
# -4 here is not "unknown" — it's structurally meaningful.
#
# Strategy:
#   - Create NEVER_SMOKED binary flag (1 = never smoked, 0 = smoked at some point)
#   - SMOKYRS, PACKSPER: recode -4 → 0 (zero years/packs is correct for non-smokers)
#   - TOBAC30, TOBAC100, QUITSMOK: recode -4 → NaN (can't assign a meaningful 0)
#   - Remaining non-(-4) sentinel codes → NaN (true missing/unknown)
 

SMOKING_COLS = ["TOBAC30", "TOBAC100", "SMOKYRS", "PACKSPER", "QUITSMOK"]
 
# Create never-smoked flag before any recoding
df["NEVER_SMOKED"] = (df["TOBAC30"] == -4).astype(int)
print(f"NEVER_SMOKED distribution:\n{df['NEVER_SMOKED'].value_counts()}\n")

# SMOKYRS, PACKSPER: -4 → 0 (zero is meaningful for non-smokers)
for col in ["SMOKYRS", "PACKSPER"]:
    if col in df.columns:
        df[col] = df[col].replace(-4, 0)
 
# TOBAC30, TOBAC100, QUITSMOK: -4 → NaN
for col in ["TOBAC30", "TOBAC100", "QUITSMOK"]:
    if col in df.columns:
        df[col] = df[col].replace(-4, np.nan)
 
# Remaining sentinel recodes for smoking variables
# SMOKYRS: 8=not assessed, 9=unknown, 88=not applicable, 99=unknown → NaN
df["SMOKYRS"]  = df["SMOKYRS"].replace([8, 9, 88, 99], np.nan)
# PACKSPER: 8=not assessed, 9=unknown → NaN
df["PACKSPER"] = df["PACKSPER"].replace([8, 9], np.nan)
# QUITSMOK: 888=unknown, 999=unknown → NaN
df["QUITSMOK"] = df["QUITSMOK"].replace([8, 9, 88, 99, 888, 999], np.nan)
# TOBAC30, TOBAC100: 9=unknown → NaN
df["TOBAC30"]  = df["TOBAC30"].replace([9], np.nan)
df["TOBAC100"] = df["TOBAC100"].replace([9], np.nan)
 
print("Smoking variable missingness after recode:")
for col in SMOKING_COLS:
    if col in df.columns:
        pct = df[col].isna().mean() * 100
        print(f"  {col}: {pct:.1f}% missing")

NEVER_SMOKED distribution:
NEVER_SMOKED
0    123532
1     71664
Name: count, dtype: int64

Smoking variable missingness after recode:
  TOBAC30: 37.0% missing
  TOBAC100: 37.4% missing
  SMOKYRS: 2.6% missing
  PACKSPER: 1.9% missing
  QUITSMOK: 75.1% missing


In [6]:
 #3. INVISITS / INCALLS — structural missingness
#
# Code 8 = "not applicable — co-participant lives with subject" (confirmed from dictionary).
# This is NOT a frequency — it means the question was skipped because proximity
# makes visit frequency irrelevant. INLIVWTH already captures this.
# Code -4 = form version did not collect this data.
# Both recoded to NaN.
 
# 
for col in ["INVISITS", "INCALLS"]:
    if col in df.columns:
        df[col] = df[col].replace([8, -4], np.nan)
 
print("INVISITS / INCALLS missingness after recode:")
for col in ["INVISITS", "INCALLS"]:
    if col in df.columns:
        pct = df[col].isna().mean() * 100
        print(f"  {col}: {pct:.1f}% missing")

INVISITS / INCALLS missingness after recode:
  INVISITS: 61.7% missing
  INCALLS: 61.7% missing


In [7]:
# 4. Remaining sentinel recodes — all other variables
#
# Each variable's sentinel codes confirmed against the NACC data dictionary.
 
# %%
SENTINEL_MAP = {
    # Demographics
    "BIRTHMO":  [8, 9],       # 8=not assessed, 9=unknown
    "NACCAGEB": [88, 99],     # 88=not applicable, 99=unknown
    "HISPANIC": [9],           # 9=unknown
    "HISPOR":   [88, 99],     # 88=not applicable (non-hispanic), 99=unknown
    "RACE":     [99],          # 99=unknown
    "RACESEC":  [88, 99],     # 88=not applicable, 99=unknown
    "RACETER":  [88, 99],     # 88=not applicable, 99=unknown
    "PRIMLANG": [8, 9],       # 8=not assessed, 9=unknown
    "HANDED":   [9],           # 9=unknown
    # Socioeconomic
    "EDUC":     [8, 9, 99],   # 8=not assessed, 9/99=unknown
    "MARISTAT": [9],           # 9=unknown
    "NACCLIVS": [9],           # 9=unknown
    "RESIDENC": [9],           # 9=unknown
    # Co-participant
    "INRELTO":  [-4],          # -4=not available
    "INLIVWTH": [-4],          # -4=not available
}

for col, sentinels in SENTINEL_MAP.items():
    if col in df.columns:
        df[col] = df[col].replace(sentinels, np.nan)
 
print("Sentinel recodes applied.")

Sentinel recodes applied.


In [8]:
# 5. Post-recode missingness profile
 

feature_cols = [c for c in df.columns if c not in [ID_COL, LABEL_COL]]
 
post_profile = pd.DataFrame({
    "n_missing":   df[feature_cols].isna().sum(),
    "pct_missing": (df[feature_cols].isna().mean() * 100).round(1),
    "n_unique":    df[feature_cols].nunique(),
}).sort_values("pct_missing", ascending=False)
 
print("\nPost-recode missingness profile:")
print(post_profile.to_string())


Post-recode missingness profile:
              n_missing  pct_missing  n_unique
RACETER          194085         99.4         6
RACESEC          189941         97.3         6
HISPOR           181905         93.2         7
QUITSMOK         146678         75.1        88
INVISITS         120391         61.7         6
INCALLS          120391         61.7         6
TOBAC100          72924         37.4         2
TOBAC30           72160         37.0         2
BIRTHMO           34276         17.6        10
INLIVWTH           8288          4.2         2
INRELTO            8288          4.2         7
SMOKYRS            5016          2.6        82
EDUC               3869          2.0        30
PACKSPER           3802          1.9         6
PRIMLANG           3032          1.6         6
RESIDENC           2878          1.5         4
NACCAGEB           1636          0.8        88
HANDED             1032          0.5         3
MARISTAT            896          0.5         6
RACE                832   

In [9]:
# 6. Baseline-only visit strategy
#
# 52,537 unique subjects; 35,671 have more than one visit (mean 3.7 visits/subject).
# Using all visits with a random row split risks leakage — the same subject can appear
# in both train and test sets, and feature values across visits are correlated.
#
# Decision: keep only the FIRST visit per subject (baseline).
# Justification: baseline visit captures non-medical risk factors at the earliest
# assessment point, before any disease progression could influence responses.
# This gives one row per subject, clean train/test split by row, and is the most
# common approach in NACC-based ML literature.

# Sort by subject ID then visit number (NACCVNUM if available, else rely on row order)
# NACCVNUM is not in our feature set so we rely on row order (first occurrence = baseline)
df_baseline = df.groupby(ID_COL, sort=False).first().reset_index()
 
print(f"\nBaseline-only dataset shape: {df_baseline.shape}")
print(f"Unique subjects: {df_baseline[ID_COL].nunique()}")
print(f"\nClass balance after baseline filter:\n"
      f"{(df_baseline[LABEL_COL].value_counts(normalize=True) * 100).round(1)}")


Baseline-only dataset shape: (52537, 27)
Unique subjects: 52537

Class balance after baseline filter:
dementia_binary
1    54.7
0    45.3
Name: proportion, dtype: float64


In [10]:
# 7. Final sanity check
 
print(f"\nFinal feature columns ({len(feature_cols)}):")
print(feature_cols)
 
print(f"\nAny remaining sentinel codes present?")
remaining_sentinels = [-4, 8, 9, 88, 99, 888, 999]
for col in feature_cols:
    if pd.api.types.is_numeric_dtype(df_baseline[col]):
        hits = {s: int((df_baseline[col] == s).sum())
                for s in remaining_sentinels
                if (df_baseline[col] == s).sum() > 0}
        if hits:
            print(f"  {col}: {hits}  <-- review against dictionary")
 
print("\nMissingness summary (baseline dataset):")
missing_summary = (df_baseline[feature_cols].isna().mean() * 100).round(1).sort_values(ascending=False)
print(missing_summary[missing_summary > 0].to_string())


Final feature columns (25):
['BIRTHMO', 'BIRTHYR', 'NACCAGEB', 'SEX', 'HISPANIC', 'HISPOR', 'RACE', 'RACESEC', 'RACETER', 'PRIMLANG', 'HANDED', 'EDUC', 'MARISTAT', 'NACCLIVS', 'RESIDENC', 'TOBAC30', 'TOBAC100', 'SMOKYRS', 'PACKSPER', 'QUITSMOK', 'INRELTO', 'INLIVWTH', 'INVISITS', 'INCALLS', 'NEVER_SMOKED']

Any remaining sentinel codes present?

Missingness summary (baseline dataset):
RACETER     99.5
RACESEC     97.1
HISPOR      91.2
QUITSMOK    62.7
INVISITS    56.3
INCALLS     56.3
BIRTHMO     17.6
INRELTO      2.8
INLIVWTH     2.8
EDUC         2.7
SMOKYRS      2.6
PACKSPER     2.0
PRIMLANG     1.8
TOBAC100     1.3
NACCAGEB     1.0
HANDED       0.8
RESIDENC     0.8
RACE         0.7
TOBAC30      0.5
HISPANIC     0.4
MARISTAT     0.4
NACCLIVS     0.2


In [11]:
# 8. Save for Step 3 (Feature Engineering)
 
out_path = os.path.join(OUTPUT_DIR, "step2_preprocessed.csv")
df_baseline.to_csv(out_path, index=False)
print(f"\nSaved: {out_path}")
print(f"Final shape: {df_baseline.shape}")


Saved: C:\Users\USER\PycharmProjects\Non-med-dementia-risk\outputs\step2_preprocessed.csv
Final shape: (52537, 27)
